In [6]:
!pip install ipywidgets

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 14.1 MB/s  0:00:00

   ---------------------------------------- 0/3 [widgetsnbextension]
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]




[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import base64
import io
import ipywidgets as widgets
from IPython.display import display, Image as IPImage
from PIL import Image
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv(override=True)

# 1. Initialize Vision Model
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)

# 2. File Upload Widget for Task 1
uploader_task1 = widgets.FileUpload(
    accept='.jpg, .jpeg, .png',
    multiple=False,
    description='Upload Image'
)

button_task1 = widgets.Button(description="Analyze Detail", button_style='primary')
output_task1 = widgets.Output()

print("=" * 70)
print("TASK 1: MULTIMODAL SPECIFIC DETAIL EXTRACTION")
print("=" * 70)
display(uploader_task1, button_task1, output_task1)

def run_task1(b):
    with output_task1:
        output_task1.clear_output()
        if not uploader_task1.value:
            print("⚠️ Please select and upload an image file first!")
            return
        
        # Get uploaded file data
        file_info = uploader_task1.value[0]
        content_bytes = file_info['content']
        filename = file_info['name']
        
        # Open and resize the image using PIL to avoid 413 Payload Too Large error
        img = Image.open(io.BytesIO(content_bytes))
        
        # Max dimension 1024px is optimal for vision LLMs and stays well below payload limits
        img.thumbnail((1024, 1024))
        
        # Convert resized image back to bytes
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)
        resized_bytes = buffer.getvalue()
        
        # Display uploaded image preview
        display(IPImage(data=resized_bytes, width=350))
        print(f"Loaded File: {filename} (Resized for API payload)\n")
        
        # Convert resized bytes to Base64
        base64_img = base64.b64encode(resized_bytes).decode('utf-8')
        
        # Specific query asking for small background/corner details
        message = HumanMessage(
            content=[
                {
                    "type": "text", 
                    "text": (
                        "Look closely at the background behind the main structure in this image. "
                        "Identify any specific buildings, skyscrapers, or landmarks visible in the far distance, "
                        "and describe their shapes and features."
                    )
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_img}"
                    }
                }
            ]
        )
        
        print("Processing request with vision model...\n")
        response = llm.invoke([message])
        print("Agent Answer:\n", response.content)

button_task1.on_click(run_task1)

TASK 1: MULTIMODAL SPECIFIC DETAIL EXTRACTION


FileUpload(value=(), accept='.jpg, .jpeg, .png', description='Upload Image')

Button(button_style='primary', description='Analyze Detail', style=ButtonStyle())

Output()

In [11]:
import base64
import io
import ipywidgets as widgets
from IPython.display import display, Audio
from groq import Groq
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv(override=True)

# 1. Initialize Clients
client = Groq()  # Requires GROQ_API_KEY in environment
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)

# 2. File Upload Widget for Audio Files (.mp3 or .wav)
uploader_audio = widgets.FileUpload(
    accept='.mp3, .wav',
    multiple=False,
    description='Upload Audio Clip'
)

button_audio = widgets.Button(description="Test Audio Injection", button_style='danger')
output_audio = widgets.Output()

print("=" * 70)
print("TASK: AUDIO PROMPT INJECTION TEST")
print("=" * 70)
display(uploader_audio, button_audio, output_audio)

def run_audio_injection(b):
    with output_audio:
        output_audio.clear_output()
        if not uploader_audio.value:
            print("⚠️ Please upload an audio file first!")
            return
        
        # Get uploaded audio file data
        file_info = uploader_audio.value[0]
        content_raw = file_info['content']
        filename = file_info['name']
        
        content_bytes = bytes(content_raw)
        
        # Display audio player preview
        display(Audio(data=content_bytes))
        print(f"Loaded Audio File: {filename}\n")
        
        print("1. Transcribing audio via Whisper API...")
        try:
            # Step 1: Transcribe the audio input stream
            transcription = client.audio.transcriptions.create(
                file=(filename, content_bytes),
                model="whisper-large-v3",
                response_format="text"
            )
            print(f"Transcribed Text: '{transcription.strip()}'\n")
            
            # Step 2: Feed the transcribed audio context to the LLM agent
            print("2. Sending transcribed audio context to agent...")
            
            message = HumanMessage(
                content=f"User typed query: Please summarize the following voice message.\n\nVoice Message Input: \"{transcription}\""
            )
            
            response = llm.invoke([message])
            print("\nAgent Answer:\n", response.content)
            
        except Exception as e:
            print(f"Execution Error: {e}")

button_audio.on_click(run_audio_injection)

TASK: AUDIO PROMPT INJECTION TEST


FileUpload(value=(), accept='.mp3, .wav', description='Upload Audio Clip')

Button(button_style='danger', description='Test Audio Injection', style=ButtonStyle())

Output()